# Part 2 — Bronze Ingestion

**Bronze** = store raw data with minimal changes.

In this notebook we:
1. read CSV files with **Spark**;
2. inspect the schema with checking the data structure;
3. add ingestion metadata;
4. save to **Delta Lake**.

**Important:** Cleaning and transformation in Silver. Filtering for ML-related tasks and feature preparation later.

In [0]:
from pyspark.sql import functions as F
from pathlib import Path

DATA_DIR = Path("../data").resolve()
FLIGHTS_CSV = f"{DATA_DIR}/flights_sample.csv"
AIRLINES_CSV = f"{DATA_DIR}/airlines.csv"
AIRPORTS_CSV = f"{DATA_DIR}/airports.csv"
BRONZE_SCHEMA = "flight_delay"
#=============================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

## 1. Read flights.csv (Spark)

In [0]:
flights_raw = (spark.read.option("header", True).option("inferSchema", True).csv(FLIGHTS_CSV))
print(f"Rows: {flights_raw.count():,}")
flights_raw.printSchema()

In [0]:
flights_raw.show(5, truncate=False)

## 2. Add ingestion metadata

Record **when** loaded (`_ingestion_timestamp`) and **which** file (`_source_file`).

In [0]:
flights_bronze = (flights_raw.withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("flights_sample.csv")))
flights_bronze.select("_ingestion_timestamp", "_source_file").show(5, truncate=False)

## 3. Save to Delta (Bronze)

**Delta Lake** — storage format with versioning(saving Bronze-level data as Delta Tables registered in Unity Catalog, not through ad-hoc path in DBFS).

In [0]:
BRONZE_FLIGHTS = f"{BRONZE_SCHEMA}.flights"
#==================================================================================
flights_bronze.write.format("delta").mode("overwrite").saveAsTable(BRONZE_FLIGHTS)
print(f"Saved: {BRONZE_FLIGHTS}")
spark.table(BRONZE_FLIGHTS).count()

## 4. Verify: read Bronze back

Reading saved Bronze table to check following:

- Number od rows matches source data;
- Ingestion metadata columns exist.

These help to ensure that the data was saved correctly.

In [0]:
flights_check = spark.table(BRONZE_FLIGHTS)
print(f"Rows in Bronze: {flights_check.count():,}")
flights_check.select("AIRLINE", "ORIGIN_AIRPORT", "DEPARTURE_DELAY", "_source_file").show(5)

In [0]:
source_count = flights_raw.count()
bronze_count = flights_check.count()

assert source_count == bronze_count, (
    f"Row count mismatch: source={source_count}, bronze={bronze_count}"
)

print(f"Validation passed: {bronze_count:,} rows")

In [0]:
required_columns = [
    "_ingestion_timestamp",
    "_source_file"
]

assert all(
    col in flights_check.columns
    for col in required_columns
)

print("Ingestion metadata validation passed")

Additional data quality check.

## 5. Bronze for lookup tables

Same ingestion process to saave airline and airport lookup tables in Bronze-level.

In [0]:
def ingest_csv_to_bronze(csv_path: str, table_name: str, source_file: str) -> str:
    dframe = (spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
        .withColumn("_ingestion_timestamp", F.current_timestamp()).withColumn("_source_file", F.lit(source_file)))
    full_table = f"{BRONZE_SCHEMA}.{table_name}"
    dframe.write.format("delta").mode("overwrite").saveAsTable(full_table)
    print(f"Saved: {full_table} ({dframe.count()} rows)")
    return full_table

BRONZE_AIRLINES = ingest_csv_to_bronze(AIRLINES_CSV, "airlines", "airlines.csv")
BRONZE_AIRPORTS = ingest_csv_to_bronze(AIRPORTS_CSV, "airports", "airports.csv")

In [0]:
spark.table(BRONZE_AIRLINES).show()
spark.table(BRONZE_AIRPORTS).show(5)

## 6. Summary

3 Bronze Delta tables created in ´flight_delay´ schema:

| Table | Name |
|-------|------|
| flights | `flight_delay.flights` |
| airlines | `flight_delay.airlines` |
| airports | `flight_delay.airports` |

**Verify:** count ≈ 200,000, `_ingestion_timestamp` and `_source_file` as metadata exist.

**Next step:** Silver - cleaning and validating data while keeping useful info for MLflow and BI.